# Police agent — parameter sensitivity study

Reproducible sensitivity analysis of the nine tunable constants in the police
policy, run against a synthetic evader on the agreed 7×7 board.

This notebook is the interactive companion to [`docs/RESEARCH.md`](../docs/RESEARCH.md).
The prose write-up carries the conclusions; this file is here so every number in it
can be re-derived, and so a different range can be tried in one edit.

**Setup:** `uv sync --extra research`, then run this notebook top to bottom.
The cached results load instantly; the live sweep at the bottom takes about a minute.


## 1. What is being measured

Each episode plays one full sub-game under the shipped rules: 7×7 board, police at
(0,0), evader at (3,3), 35 move ceiling, 14 barriers. The turn order and the
belief-update order are copied from the real runtime (`peer/turn_handler.py`), so a
sweep measures the game the agent actually plays.

Four metrics per episode:

| Metric | Meaning |
|---|---|
| `capture_rate` | fraction of episodes the police won |
| `mean_steps` | how long an episode lasted (capture speed) |
| `hit_rate` | fraction of turns the posterior's argmax was the evader's real cell |
| `mean_error` | mean Manhattan distance from the believed cell to the real one |

The evader is **omniscient and fleeing** — it always maximises distance from the
police's true cell. That is the pessimistic case: a constant that still helps here
is not winning by luck.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

DATA = Path("..") / "docs" / "research-data.json"
study = json.loads(DATA.read_text())
print(f"{study['episodes']} episodes per point, evader policy: {study['policy']}")
print("belief   :", ", ".join(study["belief"]))
print("strategy :", ", ".join(study["strategy"]))

## 2. Belief-filter constants

The five constants of the Bayesian filter (`strategy/belief_defaults.py`). These are
private tuning, not agreed terms, so we are free to choose them.


In [ ]:
def table(section, name):
    """Print one swept parameter as a readable table."""
    block = study[section][name]
    print(f"{name}  (shipped = {block['shipped']})")
    print(f"{'value':>8} {'capture':>9} {'95% CI':>16} {'steps':>7} {'hit':>7} {'error':>7}")
    for row in block["points"]:
        low, high = row["ci95"]
        mark = "  <-- shipped" if row["value"] == block["shipped"] else ""
        print(
            f"{row['value']:>8} {row['capture_rate']:>9.3f} "
            f"{f'[{low:.3f}, {high:.3f}]':>16} {row['mean_steps']:>7.1f} "
            f"{row['hit_rate']:>7.3f} {row['mean_error']:>7.2f}{mark}"
        )
    print()


for name in study["belief"]:
    table("belief", name)

### Capture rate vs belief tracking

The interesting question is whether they agree. A filter that localises better should
catch more — but that is a hypothesis, not a given, and the plots below test it.


In [ ]:
def plot(section, name, ax_pair):
    """Capture rate (with 95% CI) beside localisation error, for one parameter."""
    block = study[section][name]
    xs = [r["value"] for r in block["points"]]
    caps = [r["capture_rate"] for r in block["points"]]
    errs = [(r["ci95"][1] - r["ci95"][0]) / 2 for r in block["points"]]
    left, right = ax_pair
    left.errorbar(xs, caps, yerr=errs, marker="o", capsize=4)
    left.axvline(block["shipped"], ls="--", c="grey")
    left.set_title(f"{name}: capture rate")
    left.set_xlabel(name)
    left.grid(alpha=0.3)
    right.plot(xs, [r["mean_error"] for r in block["points"]], marker="s", c="crimson")
    right.axvline(block["shipped"], ls="--", c="grey")
    right.set_title(f"{name}: mean error")
    right.set_xlabel(name)
    right.grid(alpha=0.3)


names = list(study["belief"])
fig, axes = plt.subplots(len(names), 2, figsize=(11, 3.1 * len(names)))
for row, name in zip(axes, names, strict=True):
    plot("belief", name, row)
fig.tight_layout()

## 3. Strategy constants

The chase and barrier constants (`strategy/placement.py`, `strategy/encirclement.py`,
`strategy/barrier.py`). Each range includes the value that switches the mechanism off,
because the clearest evidence a mechanism earns its place is what happens without it.


In [ ]:
for name in study["strategy"]:
    table("strategy", name)

In [ ]:
names = list(study["strategy"])
fig, axes = plt.subplots(1, len(names), figsize=(3.6 * len(names), 3.4), squeeze=False)
for ax, name in zip(axes[0], names, strict=True):
    block = study["strategy"][name]
    xs = [r["value"] for r in block["points"]]
    caps = [r["capture_rate"] for r in block["points"]]
    errs = [(r["ci95"][1] - r["ci95"][0]) / 2 for r in block["points"]]
    ax.errorbar(xs, caps, yerr=errs, marker="o", capsize=4)
    ax.axvline(block["shipped"], ls="--", c="grey")
    ax.set_title(name, fontsize=10)
    ax.grid(alpha=0.3)
axes[0][0].set_ylabel("capture rate")
fig.tight_layout()

## 4. Run your own sweep

Everything above is cached. This re-runs one parameter live so a different range,
a different evader policy or a larger sample can be tried immediately.

Note the sample size: at 100 episodes the 95% interval on a capture rate near 0.35 is
roughly ±0.09, which is wider than most of the effects in this study. Do not read a
small live run as contradicting the cached numbers.


In [ ]:
import sys

sys.path.insert(0, "..")
from research.sweeps import sweep_strategy

points = sweep_strategy("WIDE_REACH", [0, 1, 2, 4, 12], episodes=100)
for p in points:
    low, high = p.capture_ci95
    print(f"WIDE_REACH={p.value:>3}  capture {p.capture_rate:.3f}  [{low:.3f}, {high:.3f}]")

## 5. Conclusions

See [`docs/RESEARCH.md`](../docs/RESEARCH.md) for the written analysis, including which
shipped defaults this study supports, which it questions, and what it deliberately does
not claim.
